<a href="https://colab.research.google.com/github/bautistabc/IntroDjango/blob/master/Copia_de_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [14]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

In [15]:
import os
from groq import Groq
from google.colab import userdata

In [43]:
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.", client)

Cliente de Groq inicializado correctamente. <groq.Groq object at 0x7a397e93bd50>


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [32]:
prompt = "¿Cuantos satelites existen en el sistema solar ?"

print(prompt)

¿Cuantos satelites existen en el sistema solar ?


In [33]:
models = client.models.list()

print("Modelos a los que tienes acceso:")
for model in models.data:
    print(f"- {model.id}")

Modelos a los que tienes acceso:
- groq/compound-mini
- openai/gpt-oss-safeguard-20b
- canopylabs/orpheus-v1-english
- qwen/qwen3.6-27b
- openai/gpt-oss-20b
- openai/gpt-oss-120b
- meta-llama/llama-prompt-guard-2-86m
- groq/compound
- canopylabs/orpheus-arabic-saudi
- whisper-large-v3
- whisper-large-v3-turbo
- allam-2-7b
- meta-llama/llama-prompt-guard-2-22m


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [35]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [36]:
print(response.model_dump_json(indent=2))

NameError: name 'response' is not defined

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [37]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)

NameError: name 'response' is not defined

### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [38]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
inicio = time.time()
response_grande = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [48]:
import os
import time
from groq import Groq
from google.colab import userdata

In [45]:
# Leer API key desde Colab Secrets
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.", client)

Cliente de Groq inicializado correctamente. <groq.Groq object at 0x7a397eb163f0>


In [46]:
# Definir la lista de preguntas
preguntas = ["Cual es la diferencia entre 4G y 5G.",
              "Menciona las principales caracteristicas de IoT",
             "Cuales son las caracteristicas de la FO."]

**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
pregunta_1 = preguntas [0]
modelo_ligero = "llama-3.1-8b-instant"

# Registro  tiempo de inicio
inicio = time.perf_counter()

response = client.chat.completions.create(
    model=modelo_ligero,
    messages=[{"role": "user", "content": pregunta_1}])

# Registro del tiempo de fin y calcular latencia
fin = time.perf_counter()
tiempo_respuesta = fin - inicio

# Se Guarda en el diccionario
resultado_1 = {
    "respuesta:": response.choices[0].message.content,
    "tiempo_respuesta_sec:": round(tiempo_respuesta, 3),
    "Tokens del prompt:": response.usage.prompt_tokens,
    "Tokens de la respuesta:": response.usage.completion_tokens,
    "Tokens totales:": response.usage.total_tokens
}



**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
pregunta_2 = preguntas [1]
#modelo_ligero = "llama-3.1-8b-instant"

# Registro  tiempo de inicio
inicio = time.perf_counter()

response = client.chat.completions.create(
    model=modelo_ligero,
    messages=[{"role": "user", "content": pregunta_2}])

# Registro del tiempo de fin y calcular latencia
fin = time.perf_counter()
tiempo_respuesta = fin - inicio

# Se Guarda en el diccionario
resultado_2 = {
    "respuesta:": response.choices[0].message.content,
    "tiempo_respuesta_sec:": round(tiempo_respuesta, 3),
    "Tokens del prompt:": response.usage.prompt_tokens,
    "Tokens de la respuesta:": response.usage.completion_tokens,
    "Tokens totales:": response.usage.total_tokens
}

In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
pregunta_3 = preguntas [2]
#modelo_ligero = "llama-3.1-8b-instant"

# Registro  tiempo de inicio
inicio = time.perf_counter()

response = client.chat.completions.create(
    model=modelo_ligero,
    messages=[{"role": "user", "content": pregunta_3}])

# Registro del tiempo de fin y calcular latencia
fin = time.perf_counter()
tiempo_respuesta = fin - inicio

# Se Guarda en el diccionario
resultado_3 = {
    "respuesta:": response.choices[0].message.content,
    "tiempo_respuesta_sec:": round(tiempo_respuesta, 3),
    "Tokens del prompt:": response.usage.prompt_tokens,
    "Tokens de la respuesta:": response.usage.completion_tokens,
    "Tokens totales:": response.usage.total_tokens
}

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados = [resultado_1, resultado_2, resultado_3]

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados
print(resultados)

Sí, el modelo ligero resolvió las 3 preguntas satisfactoriamente